In [ ]:
from google.colab import drive, userdata
import os, sys

drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/DataScience/molecular-ai-agent'
os.makedirs(PROJECT_DIR, exist_ok=True)
os.chdir(PROJECT_DIR)
sys.path.append(PROJECT_DIR)

In [ ]:
!pip install torch-geometric -q
!pip install torch-scatter torch-sparse \
    -f https://data.pyg.org/whl/torch-2.0.0+cu118.html -q
!pip install rdkit -q

# Verify
import torch
import torch_geometric
from rdkit import Chem

print(f"PyTorch          : {torch.__version__}")
print(f"PyTorch Geometric: {torch_geometric.__version__}")
print(f"CUDA available   : {torch.cuda.is_available()}")
print(f"GPU              : {torch.cuda.get_device_name(0)}")
print("All packages ready.")

In [ ]:
%%writefile /content/drive/MyDrive/DataScience/molecular-ai-agent/src/utils/mol_utils.py

from rdkit import Chem
from rdkit.Chem import Draw, Descriptors
import numpy as np

def get_atom_features(atom):
    atomic_num   = atom.GetAtomicNum()
    degree       = atom.GetDegree()
    degree_onehot = [int(degree == i) for i in range(5)]
    formal_charge = atom.GetFormalCharge()
    num_hs       = atom.GetTotalNumHs()
    num_radical   = atom.GetNumRadicalElectrons()
    in_ring      = int(atom.IsInRing())
    is_aromatic  = int(atom.GetIsAromatic())

    return [atomic_num] + degree_onehot + [formal_charge, num_hs, num_radical, in_ring, is_aromatic]

def is_valid_smiles(smiles: str) -> bool:
    return Chem.MolFromSmiles(smiles) is not None

def smiles_to_features(smiles: str):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None, None, False

    mol = Chem.AddHs(mol)

    node_features = [get_atom_features(atom) for atom in mol.GetAtoms()]

    edge_index = []
    for bond in mol.GetBonds():
        i = bond.GetBeginAtomIdx()
        j = bond.GetEndAtomIdx()
        edge_index.append([i, j])
        edge_index.append([j, i])

    return node_features, edge_index, True

def get_basic_descriptors(smiles: str) -> dict:
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return {}
    return {
        "molecular_weight"  : round(Descriptors.MolWt(mol), 3),
        "num_atoms"         : mol.GetNumAtoms(),
        "num_rings"         : mol.GetRingInfo().NumRings(),
        "logP"              : round(Descriptors.MolLogP(mol), 3),
        "num_h_donors"      : Descriptors.NumHDonors(mol),
        "num_h_acceptors"   : Descriptors.NumHAcceptors(mol),
    }

def draw_molecule(smiles: str, size=(300, 300)):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return Draw.MolToImage(mol, size=size)

In [ ]:
import sys
sys.path.append('/content/drive/MyDrive/DataScience/molecular-ai-agent')

from src.utils.mol_utils import (
    is_valid_smiles,
    smiles_to_features,
    get_basic_descriptors,
    draw_molecule
)

# caffeine SMILES
smiles = "CN1C=NC2=C1C(=O)N(C(=O)N2C)C"
print("Valid SMILES:", is_valid_smiles(smiles))

# feature extraction
nodes, edges, valid = smiles_to_features(smiles)
print(f"Atoms: {len(nodes)}, Bonds (directed): {len(edges)}")
print(f"First atom features: {nodes[0]}")

# descriptors
desc = get_basic_descriptors(smiles)
print("Descriptors:", desc)

# invalid SMILES
print("Invalid SMILES:", is_valid_smiles("NOTASMILES###"))

In [5]:
from rdkit import Chem

_original_supplier = Chem.SDMolSupplier

class SafeSupplier(_original_supplier):
  def __iter__(self):
      for mol in super().__iter__():
        if mol is None:
          continue
        yield mol
Chem.SDMolSupplier = SafeSupplier

In [ ]:
from torch_geometric.datasets import QM9

def valid_mol(data):
  return data.pos is not None

dataset = QM9(root=f'/content/drive/MyDrive/DataScience/molecular-ai-agent/data/qm9', pre_filter = valid_mol)

print(f"Total molecules  : {len(dataset)}")
print(f"Target properties: {dataset.num_classes}")

# example
mol = dataset[0]
print(f"\nSingle molecule:")
print(f"  Atom features  : {mol.x.shape}")
print(f"  Edge index     : {mol.edge_index.shape}")
print(f"  Properties     : {mol.y.shape}")
print(f"  HOMO-LUMO gap  : {mol.y[0, 4].item():.4f} Hartree")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

os.makedirs(f'/content/drive/MyDrive/DataScience/molecular-ai-agent/plots', exist_ok=True)

# HOMO-LUMO gap distribution
gaps = dataset.data.y[:, 4].numpy()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(gaps, bins=80, color='#7F77DD', alpha=0.85, edgecolor='none')
axes[0].set_xlabel('HOMO-LUMO Gap (Hartree)')
axes[0].set_ylabel('Count')
axes[0].set_title('HOMO-LUMO Gap Distribution across QM9')

axes[1].boxplot(gaps, patch_artist=True,
                boxprops=dict(facecolor='#7F77DD', alpha=0.6))
axes[1].set_ylabel('HOMO-LUMO Gap (Hartree)')
axes[1].set_title('Outlier check')

plt.tight_layout()
plt.savefig(f'/content/drive/MyDrive/DataScience/molecular-ai-agent/plots/gap_distribution.png', dpi=150)
plt.show()

print(f"Mean : {gaps.mean():.4f}")
print(f"Std  : {gaps.std():.4f}")
print(f"Min  : {gaps.min():.4f}")
print(f"Max  : {gaps.max():.4f}")